In [1]:
from pathlib import Path
import re
import os
import h5py
import numpy as np

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
OUTDIR = Path("/ceph/dwong/work/training_samples/NR")
ENERGY = 1000
DELETE_BATCHES = False  # set False if you want to keep the small files

# --------------------------------------------------
# MERGE ZST FILES
# --------------------------------------------------
print(f"=== Merging ZST batches for energy={ENERGY} ===")

pattern_zst = re.compile(rf"^traces_energy_{ENERGY}_batch_(\d{{4}})\.zst$")
batch_files = sorted(
    (int(m.group(1)), p)
    for p in OUTDIR.iterdir()
    if (m := pattern_zst.match(p.name))
)
if not batch_files:
    raise RuntimeError(f"No ZST batches found for energy={ENERGY}")

merged_zst = OUTDIR / f"traces_energy_{ENERGY}.zst"
with open(merged_zst, "wb") as fout:
    for idx, p in batch_files:
        with open(p, "rb") as fin:
            while chunk := fin.read(1024 * 1024):
                fout.write(chunk)
        if DELETE_BATCHES:
            p.unlink()

print(f"✔ merged {len(batch_files)} trace batches → {merged_zst.name}")

# --------------------------------------------------
# MERGE H5 META FILES
# --------------------------------------------------
print(f"=== Merging HDF5 meta batches for energy={ENERGY} ===")

pattern_h5 = re.compile(rf"^meta_energy_{ENERGY}_batch_(\d{{4}})\.h5$")
meta_files = sorted(
    (int(m.group(1)), p)
    for p in OUTDIR.iterdir()
    if (m := pattern_h5.match(p.name))
)
if not meta_files:
    raise RuntimeError(f"No HDF5 meta batches found for energy={ENERGY}")

# Read dtype & attrs from first batch
first_path = meta_files[0][1]
with h5py.File(first_path, "r") as f0:
    n_channels = int(f0.attrs["n_channels"])
    trace_samples = int(f0.attrs["trace_samples"])
    trace_dtype = f0.attrs["trace_dtype"]
    meta_dtype = f0["events"].dtype

# Count total rows
total_events = 0
for _, p in meta_files:
    with h5py.File(p, "r") as fin:
        total_events += fin["events"].shape[0]

merged_meta = OUTDIR / f"meta_energy_{ENERGY}.h5"
with h5py.File(merged_meta, "w") as fout:
    dset = fout.create_dataset("events", shape=(total_events,), dtype=meta_dtype, chunks=True)
    fout.attrs["n_channels"] = n_channels
    fout.attrs["trace_samples"] = trace_samples
    fout.attrs["trace_dtype"] = trace_dtype
    fout.attrs["compression"] = "zstd"

    offset = 0
    for idx, p in meta_files:
        with h5py.File(p, "r") as fin:
            arr = fin["events"][:]
            n = len(arr)
            dset[offset:offset+n] = arr
            offset += n
        if DELETE_BATCHES:
            p.unlink()

print(f"✔ merged {len(meta_files)} meta batches ({total_events} events) → {merged_meta.name}")
print("All done ✅")


The history saving thread hit an unexpected error (DatabaseError('database disk image is malformed')).History will not be written to the database.
=== Merging ZST batches for energy=1000 ===
✔ merged 25 trace batches → traces_energy_1000.zst
=== Merging HDF5 meta batches for energy=1000 ===


OSError: Unable to open file (truncated file: eof = 96, sblock->base_addr = 0, stored_eof = 2048)

In [2]:
from pathlib import Path
import h5py, re

OUTDIR = Path("/ceph/dwong/work/training_samples/NR")
ENERGY = 1000

pat = re.compile(rf"^meta_energy_{ENERGY}_batch_(\d{{4}})\.h5$")
good, bad = [], []

for p in sorted(OUTDIR.iterdir()):
    m = pat.match(p.name)
    if not m: 
        continue
    try:
        with h5py.File(p, "r") as f:
            ok = ("events" in f and f["events"].shape[0] > 0)
    except Exception:
        ok = False
    (good if ok else bad).append((int(m.group(1)), p))

print("Good meta batches:", [i for i,_ in good])
print("Bad meta batches :", [i for i,_ in bad])


Good meta batches: []
Bad meta batches : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
